# E-Commerce Data Cleaning & Dimensional Modeling
**Author:** Piyush Kumar Singh
**Goal:** Clean raw transactions and normalize into a Star Schema (Dimensions & Facts).

In [1]:
import os
import pandas as pd

## 1. Load Raw Data

In [2]:
raw_path = 'data/raw/raw_ecommerce_transactions.csv'
df_raw = pd.read_csv(raw_path)
if 'unit_price_x' in df_raw.columns:
    df_raw = df_raw.rename(columns={'unit_price_x': 'unit_price'})

print(f'Total raw records: {len(df_raw):,}')
df_raw.head()

Total raw records: 69,793


,item_id,order_id,product_id,quantity,unit_price,total_item_price,order_date,customer_id,payment_method,total_amount,customer_name,email,city,state,product_name,category,unit_price_y
0,ITEM-0000001,ORD-000001,PROD-25,3,399.0,1197.0,2024-01-16,CUST-01936,Debit Card,1197.0,Pooja Kumar,pooja.kumar77@example.com,Bangalore,Karnataka,Insulated Lunch Bag,Home & Kitchen,399.0
1,ITEM-0000002,ORD-000002,PROD-06,1,1999.0,1999.0,2024-04-15,CUST-02409,Credit Card,3796.0,Pooja Gupta,pooja.gupta569@example.com,Bangalore,Karnataka,Smart Fitness Band,Wearables,1999.0
2,ITEM-0000003,ORD-000002,PROD-07,3,599.0,1797.0,2024-04-15,CUST-02409,Credit Card,3796.0,Pooja Gupta,pooja.gupta569@example.com,Bangalore,Karnataka,Stainless Steel Water Bottle,Home & Kitchen,599.0
3,ITEM-0000004,ORD-000003,PROD-01,2,499.0,998.0,2024-05-21,CUST-03796,Cash on Delivery,998.0,Rohan Patel,rohan.patel870@example.com,Delhi,Delhi,Wireless Optical Mouse,Electronics,499.0
4,ITEM-0000005,ORD-000004,PROD-04,1,299.0,299.0,2024-03-27,CUST-08822,UPI,698.0,Rahul Patel,rahul.patel41@example.com,Hyderabad,Telangana,USB-C Fast Charging Cable,Accessories,299.0


## 2. Data Cleaning

In [3]:
# Remove missing customer IDs
df_clean = df_raw.dropna(subset=['customer_id'])

# Filter out negative or zero prices
df_clean = df_clean[df_clean['total_amount'] > 0]

# Remove duplicate rows
df_clean = df_clean.drop_duplicates()

print(f'Initial rows: {len(df_raw):,}')
print(f'Clean rows: {len(df_clean):,}')
print(f'Rejected rows: {len(df_raw) - len(df_clean):,}')

Initial rows: 69,793
Clean rows: 69,787
Rejected rows: 6


## 3. Split into Star Schema (Dimensions & Facts)

In [4]:
# 1. Customer Dimension
df_customers = df_clean[['customer_id', 'customer_name', 'email', 'city', 'state']].drop_duplicates(subset=['customer_id'])

# 2. Product Dimension
df_products = df_clean[['product_id', 'product_name', 'category', 'unit_price']].drop_duplicates(subset=['product_id'])

# 3. Fact Orders
df_orders = df_clean[['order_id', 'order_date', 'customer_id', 'payment_method', 'total_amount']].drop_duplicates(subset=['order_id'])

# 4. Fact Order Items
df_items = df_clean[['item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'total_item_price']]

print(f'dim_customers: {len(df_customers):,} rows')
print(f'dim_products: {len(df_products):,} rows')
print(f'fact_orders: {len(df_orders):,} rows')
print(f'fact_order_items: {len(df_items):,} rows')

dim_customers: 11,684 rows
dim_products: 25 rows
fact_orders: 44,996 rows
fact_order_items: 69,787 rows


## 4. Export Cleaned Data

In [5]:
clean_dir = 'data/cleaned'
os.makedirs(clean_dir, exist_ok=True)

df_customers.to_csv(f'{clean_dir}/dim_customers.csv', index=False)
df_products.to_csv(f'{clean_dir}/dim_products.csv', index=False)
df_orders.to_csv(f'{clean_dir}/fact_orders.csv', index=False)
df_items.to_csv(f'{clean_dir}/fact_order_items.csv', index=False)

print('Successfully exported all cleaned tables to data/cleaned/')

Successfully exported all cleaned tables to data/cleaned/


## 5. Summary Business KPIs

In [6]:
gross_revenue = df_orders['total_amount'].sum()
total_orders = len(df_orders)
aov = gross_revenue / total_orders

print(f'Total Gross Revenue: Rs. {gross_revenue:,.2f}')
print(f'Total Completed Orders: {total_orders:,}')
print(f'Average Order Value (AOV): Rs. {aov:.2f}')

Total Gross Revenue: Rs. 100,798,283.00
Total Completed Orders: 44,996
Average Order Value (AOV): Rs. 2240.16
